# Taxonomy Metadata generation
Extracting taxonomy information from `IAMM_metadata_file.tsv` and aligning strain IDs with `iamm_references.csv` to generate a new taxonomy metadata file, allowing for cleanup of old files.

In [32]:
# import libraries
import pandas as pd

In [33]:
# Load in files
reference_df = pd.read_csv('./iamm_references.csv', index_col = 0)
old_meta_df = pd.read_table('./IAMM_metadata_file.tsv', index_col = 0)

In [34]:
# Compare indices
reference_missing_meta = reference_df.index.difference(old_meta_df.index)
reference_df.loc[reference_missing_meta]

,ref_genome_name
atcc25922,GCF_000401755.1_Escherichia_coli_ATCC_25922_ge...
atcc26499,2545824627
atcc27126,GCA_000172635
jeju,2684622907
mr1,GCF_000146165.2_ASM14616v2_genomic
psych,GCF_000381745.1_ASM38174v1_genomic


In [35]:
meta_missing_reference = old_meta_df.index.difference(reference_df.index)
old_meta_df.loc[meta_missing_reference]

,phylum,class,order,family,genus,species,strain_designation,source,source_catalog_number,id_RES,...,LIB_Source,LIB_Source_catalog_number,LIB_Note,LIB_Strain_nickname,ZOC_species,ZOC_filename,ZOC_GFC,ZOC_repository,ZOC_isolation_source,ZOC_isolation_notes
23913,NaN,NaN,NaN,NaN,Sinorhizobium,meliloti,23913,NaN,NaN,D20-160082-4500T,...,NaN,NaN,NaN,NaN,Sinorhizobium meliloti DSM 23913,2597490345,14.0,JGI,PA,Aral sea region
25922,Proteobacteria,Gammaproteobacteria,Enterobacteriales,Enterobacteriaceae,Escherichia,coli,Seattle 1946,ATCC,25922,NaN,...,ATCC,25922,NaN,25922,NaN,NaN,NaN,NaN,NaN,NaN
26499,Proteobacteria,Gammaproteobacteria,Alteromonadales,Alteromonadaceae,Alteromonas,mediterranea,MED64,DSMZ,26499,D20-160031-4500T,...,DSMZ,26499,NaN,26499,Alteromonas macleodii 'Aegean Sea MED64',2545824627,24.0,JGI,SW,Coastal waters of Israel (Eastern Aegean)
27126,Proteobacteria,Gammaproteobacteria,Alteromonadales,Alteromonadaceae,Alteromonas,macleodii,107,ATCC,27126,D20-160027-4500T,...,ATCC,27126,NaN,27126,Alteromonas macleodii ATCC 27126,GCA_000172635,24.0,NCBI,SW,seawater
dsm100870,NaN,NaN,NaN,NaN,Poseidonibacter,lekithochrous,DSM 100870,NaN,NaN,D20-160086-4500T,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dsm15273,NaN,NaN,NaN,NaN,Serinicoccus,marinus,DSM 15273,NaN,NaN,D20-160072-4500T,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dsm15293,NaN,NaN,NaN,NaN,Halomonas,campaniensis,DSM 15293,NaN,NaN,D20-160083-4500T,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dsm16067,NaN,NaN,NaN,NaN,Algoriphagus,marincola,DSM 16067,NaN,NaN,D20-160076-4500T,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dsm17348,NaN,NaN,NaN,NaN,Kocuria,sp.,DSM 17348,NaN,NaN,D20-160071-4500T,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dsm17526,NaN,NaN,NaN,NaN,Echinicola,vietnamensis,DSM 17526,NaN,NaN,D20-160074-4500T,...,NaN,NaN,NaN,NaN,"Echinicola vietnamensis KMM 6221, DSM 17526",2509276020,6.0,JGI,SW,"Seawater collected in a musselfarm; Vietnam, S..."


In [36]:
# Remap and drop old indices and irrelevant columns
mapper = {
  '25922': 'atcc25922',
  '26499': 'atcc26499',
  '27126': 'atcc27126',
}
rows_to_drop = meta_missing_reference.drop(mapper.keys())
new_tax_df = old_meta_df.drop(index = rows_to_drop)
new_tax_df = new_tax_df[['phylum', 'class', 'order', 'family', 'genus', 'species', 'strain_designation']]
new_tax_df.index = new_tax_df.index.map(lambda k: mapper[k] if k in mapper else k).rename('strain_id')
new_tax_df.index.symmetric_difference(reference_df.index)

Index(['jeju', 'mr1', 'psych'], dtype='object', name='strain_id')

In [37]:
# Add missing rows and determine which additional rows have missing data
for missing_strain in new_tax_df.index.symmetric_difference(reference_df.index):
  new_tax_df.loc[missing_strain] = None

new_tax_df.loc[new_tax_df.drop(columns = ['strain_designation']).isna().any(axis = 1)]

,phylum,class,order,family,genus,species,strain_designation
strain_id,,,,,,,
pr1red,NaN,NaN,NaN,NaN,Algoriphagus,machipongonensis,Pr1red
parctic,NaN,NaN,NaN,NaN,NaN,NaN,NaN
plank,NaN,NaN,NaN,NaN,NaN,NaN,NaN
jeju,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mr1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
psych,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
# Fill in missing data
new_tax_df.loc['pr1red', ['phylum', 'class', 'order', 'family']] = ['Bacteroidetes', 'Cytophagia', 'Cytophagales', 'Cyclobacteriaceae']
new_tax_df.loc['parctic'] = ['Proteobacteria', 'Gammaproteobacteria', 'Alteromonadales', 'Psychromonadaceae', 'Psychromonas', 'arctica', 'DSM 14288']
new_tax_df.loc['plank'] = ['Proteobacteria', 'Alphaproteobacteria', 'Rhodobacterales', 'Rhodobacteraceae', 'Planktomarina', 'temperata', 'DSM 22400']
new_tax_df.loc['jeju'] = ['Bacteroidetes', 'Flavobacteriia', 'Flavobacteriales', 'Flavobacteriaceae', 'Hyunsoonleella', 'jejuensis', 'DSM 21035']
new_tax_df.loc['mr1'] = ['Proteobacteria', 'Gammaproteobacteria', 'Alteromonadales', 'Shewanellaceae', 'Shewanella', 'oneidensis', 'MR-1']
new_tax_df.loc['psych'] = ['Proteobacteria', 'Gammaproteobacteria', 'Alteromonadales', 'Psychromonadaceae', 'Psychromonas', 'ossibalaenae', 'JAMM 0738']

In [39]:
# Replace "none" in strain_designation with actual None value (translates to empty column in output for easier concatenation)
none_strains = new_tax_df.index[new_tax_df['strain_designation'] == 'none']
new_tax_df.loc[none_strains, 'strain_designation'] = None

In [40]:
new_tax_df.to_csv('iamm_taxonomy.csv')